In [19]:
import pandas as pd
import torch
import torch.nn.functional as F
import joblib

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [20]:
MODEL_DIR = "../models/transformer_classifier_dataset_1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

transformer = (
    AutoModelForSequenceClassification
    .from_pretrained(MODEL_DIR)
)

transformer.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,

In [21]:
dataset = load_from_disk("../data/processed_dataset_1")

test_dataset = dataset["test"]

print(len(test_dataset))

2000


In [22]:
transformer_errors = []

for sample in test_dataset:

    text = sample["text"]
    true_label = sample["label"]

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = transformer(**inputs)

    probs = F.softmax(
        outputs.logits,
        dim=-1
    )[0]

    pred_label = probs.argmax().item()

    confidence = probs[pred_label].item()

    if pred_label != true_label:

        transformer_errors.append({
            "text": text,
            "true_label": true_label,
            "predicted_label": pred_label,
            "confidence_score": confidence
        })

print(
    "Transformer Errors:",
    len(transformer_errors)
)

Transformer Errors: 44


In [24]:
lr_model = joblib.load(
    "../models/classical_baselines/logistic_regression.pkl"
)

vectorizer = joblib.load(
    "../models/classical_baselines/tfidf_vectorizer.pkl"
)

In [ ]:
from pathlib import Path

transformer_df = pd.DataFrame(
    transformer_errors
)

transformer_df = transformer_df.sort_values(
    "confidence_score",
    ascending=False
)

output_dir = Path(
    "../results/error_analysis"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transformer_df.to_csv(
    output_dir / "wrong_predictions.csv",
    index=False
)

print(
    "Saved transformer errors"
)

Saved transformer errors


In [29]:
top20 = transformer_df.head(20)

top20[
    [
        "text",
        "true_label",
        "predicted_label",
        "confidence_score"
    ]
]

,text,true_label,predicted_label,confidence_score
13,404 plan on wednesday at 12 - 00 do you have y...,1,0,0.996354
12,movie project hi steph\ni have found the talen...,1,0,0.995727
8,get a $ 25 certificate just for responding to ...,0,1,0.995639
42,new email yo bro . this is my new email addres...,0,1,0.995337
38,thank you for renewing your subscription to po...,0,1,0.995241
41,"michael romero tue , 21 jun 2005 07 : 33 : 23 ...",1,0,0.994930
21,"cdnow shipment confirmation dear daren ,\nthan...",0,1,0.994739
16,exclusive brands www . w 6 iqzkyko 7 e 6 xhe ....,1,0,0.994556
23,fw : re : he rbv i agr a see the . htm attachm...,1,0,0.994345
28,can ' t please everyone thought you might enjo...,0,1,0.994091


In [ ]:
! pip install -U joblib


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import joblib
lr_model = joblib.load(
    "../models/classical_baselines/logistic_regression.pkl"
)

vectorizer = joblib.load(
    "../models/classical_baselines/tfidf_vectorizer.pkl"
)

In [35]:
test_dataset = dataset["test"]

In [36]:
X_test = vectorizer.transform(
    test_dataset["text"]
)

predictions = lr_model.predict(X_test)

probabilities = lr_model.predict_proba(X_test)

baseline_errors = []

for i in range(len(test_dataset)):

    text = test_dataset[i]["text"]

    true_label = test_dataset[i]["label"]

    pred_label = predictions[i]

    confidence = (
        probabilities[i].max()
    )

    if pred_label != true_label:

        baseline_errors.append({
            "text": text,
            "true_label": true_label,
            "predicted_label": pred_label,
            "confidence_score": confidence
        })

print(
    "Baseline Errors:",
    len(baseline_errors)
)

Baseline Errors: 25


In [41]:
baseline_df = pd.DataFrame(
    baseline_errors
)
baseline_df = baseline_df.sort_values(
    "confidence_score",
    ascending=False
)

baseline_df.to_csv(
    output_dir /
    "logistic_wrong_predictions.csv",
    index=False
)

In [42]:
top20 = baseline_df.head(20)

top20[
    [
        "text",
        "true_label",
        "predicted_label",
        "confidence_score"
    ]
]


,text,true_label,predicted_label,confidence_score
12,nymex invitation - learn power trading power t...,1,0,0.950910
1,nymex invitation - learn power trading power t...,1,0,0.950910
23,new email yo bro . this is my new email addres...,0,1,0.924208
21,can ' t please everyone thought you might enjo...,0,1,0.906091
6,get a $ 25 certificate just for responding to ...,0,1,0.904134
9,i ' m out bill - i ' m in copenhagen awaiting ...,0,1,0.892559
0,re : buzzwords also please search for the name...,0,1,0.859487
15,si is back up si is back up in folsom,0,1,0.764344
13,invitation to dinner music can be started by c...,0,1,0.753682
8,marketers we are dropping a lot of marketers ....,0,1,0.705673


In [45]:
import os
import pandas as pd
from collections import Counter
from datasets import load_from_disk

# =====================================================
# Paths
# =====================================================

notebook_dir = os.path.abspath('')
project_root = os.path.dirname(notebook_dir)

ERRORS_FILE = os.path.join(
    project_root, 
    "results", 
    "error_analysis", 
    "wrong_predictions.csv"
)

DATASET_PATH = os.path.join(
    project_root, 
    "data", 
    "processed_dataset_1"
)

# =====================================================
# Load Data
# =====================================================

print("Loading data for deeper analysis...\n")

wrong_preds = pd.read_csv(ERRORS_FILE)
dataset = load_from_disk(DATASET_PATH)

# Convert training data to DataFrame to analyze training volume
train_df = pd.DataFrame(dataset["train"])

# Ensure text length is calculated
wrong_preds['text_length'] = wrong_preds['text'].apply(lambda x: len(str(x).split()))
train_df['text_length'] = train_df['text'].apply(lambda x: len(str(x).split()))

num_classes = len(train_df['label'].unique())

print("="*50)
print("ERROR PATTERN DIAGNOSTICS")
print("="*50)

# =====================================================
# 1 & 4. Short Text vs Long Text
# =====================================================

short_threshold = 5   # Less than 5 words
long_threshold = 100  # More than 100 words

short_errors = len(wrong_preds[wrong_preds['text_length'] <= short_threshold])
long_errors = len(wrong_preds[wrong_preds['text_length'] >= long_threshold])

print(f"1. Short Text Errors (<= {short_threshold} words): {short_errors} examples")
print(f"2. Long Text Errors (>= {long_threshold} words): {long_errors} examples\n")

# =====================================================
# 2. Ambiguous Text
# =====================================================
# If confidence is close to a random guess (e.g., 0.5 for 2 classes), 
# the model was genuinely confused/ambiguous.

random_guess_prob = 1.0 / num_classes
ambiguous_errors = len(wrong_preds[
    (wrong_preds['confidence_score'] > (random_guess_prob - 0.15)) & 
    (wrong_preds['confidence_score'] < (random_guess_prob + 0.15))
])

print(f"3. Ambiguous Text Errors (Confidence near {random_guess_prob:.2f}): {ambiguous_errors} examples\n")

# =====================================================
# 3. Rare Words
# =====================================================
# Find words that appear less than 5 times in the training data

all_train_words = " ".join(train_df['text'].astype(str)).split()
word_counts = Counter(all_train_words)

# Set of rare words
rare_words = set(word for word, count in word_counts.items() if count < 5)

def get_rare_word_ratio(text):
    words = str(text).split()
    if not words: return 0
    rare_count = sum(1 for w in words if w in rare_words)
    return rare_count / len(words)

wrong_preds['rare_word_ratio'] = wrong_preds['text'].apply(get_rare_word_ratio)

# Flag if more than 15% of the text consists of rare words
rare_word_errors = len(wrong_preds[wrong_preds['rare_word_ratio'] > 0.15])

print(f"4. Rare Word Errors (>15% rare vocabulary): {rare_word_errors} examples\n")

# =====================================================
# 5. Similar Class Labels
# =====================================================

confused = wrong_preds.groupby(['true_label', 'predicted_label']).size().reset_index(name='count')
confused = confused.sort_values(by='count', ascending=False)

print("5. Similar Class Labels (Top Confused Pairs):")
for index, row in confused.head(3).iterrows():
    print(f"   - True Label '{row['true_label']}' predicted as '{row['predicted_label']}' ({row['count']} times)")
print()

# =====================================================
# 6. Insufficient Training Examples
# =====================================================

print("6. Insufficient Training Examples (Imbalance Check):")
train_dist = train_df['label'].value_counts(normalize=True) * 100
error_dist = wrong_preds['true_label'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Train_Percentage': train_dist, 
    'Error_Percentage': error_dist
}).fillna(0)

print(dist_df.round(1))

Loading data for deeper analysis...

ERROR PATTERN DIAGNOSTICS
1. Short Text Errors (<= 5 words): 0 examples
2. Long Text Errors (>= 100 words): 25 examples

3. Ambiguous Text Errors (Confidence near 0.50): 5 examples

4. Rare Word Errors (>15% rare vocabulary): 2 examples

5. Similar Class Labels (Top Confused Pairs):
   - True Label '0' predicted as '1' (23 times)
   - True Label '1' predicted as '0' (21 times)

6. Insufficient Training Examples (Imbalance Check):
   Train_Percentage  Error_Percentage
0              49.1              52.3
1              50.9              47.7


## Step 11.4: Common Error Patterns Analysis

In [47]:
# same_wrong_examples.py

import pandas as pd

# transformer errors
transformer_errors = pd.read_csv(
    "../results/error_analysis/wrong_predictions.csv"
)

# logistic regression errors
baseline_errors = pd.read_csv(
    "../results/error_analysis/logistic_wrong_predictions.csv"
)

# Find common wrong examples using text
common_errors = pd.merge(
    transformer_errors,
    baseline_errors,
    on="text",
    suffixes=(
        "_transformer",
        "_baseline"
    )
)

print(
    f"Wrong by Transformer: "
    f"{len(transformer_errors)}"
)

print(
    f"Wrong by Baseline: "
    f"{len(baseline_errors)}"
)

print(
    f"Wrong by Both: "
    f"{len(common_errors)}"
)

print("\nExamples wrong by both:\n")

print(
    common_errors[
        [
            "text",
            "true_label_transformer",
            "predicted_label_transformer",
            "predicted_label_baseline"
        ]
    ].head(20)
)

Wrong by Transformer: 44
Wrong by Baseline: 25
Wrong by Both: 7

Examples wrong by both:

                                                text  true_label_transformer  \
0  get a $ 25 certificate just for responding to ...                       0   
1  new email yo bro . this is my new email addres...                       0   
2  can ' t please everyone thought you might enjo...                       0   
3             mirant and koch arent selling physical                       0   
4  i ' m out bill - i ' m in copenhagen awaiting ...                       0   
5  spring savings certificate - take 30 % off sav...                       0   
6              si is back up si is back up in folsom                       0   

   predicted_label_transformer  predicted_label_baseline  
0                            1                         1  
1                            1                         1  
2                            1                         1  
3                            1   

## Step 11.5: Compare Baseline vs Transformer Model Errors